In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Purane packages hata kar naye install karein
!pip uninstall -q pytorchvideo -y
!pip install -q git+https://github.com/facebookresearch/pytorchvideo.git
!pip install -q torchvision av decord
!pip install pytorchvideo --quiet

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 MB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 108.7 MB/s eta 0:00:00


In [ ]:
# ===============================================
# SlowFast Video Classification: Fight vs Non-Fight
# GitHub / Resume Ready Final Version ✅
#
# Improvements Added:
# - Fixed validation indentation
# - Shuffle enabled
# - AdamW optimizer + weight decay
# - Lower learning rate
# - Dropout to reduce overfitting
# - Early stopping
# - Best + Last model saving
# ===============================================

import torch
from torch.utils.data import DataLoader
from torchvision.transforms import Compose, Lambda, CenterCrop
import sys

# --- Torchvision Compatibility Fix ---
try:
    import torchvision.transforms.functional_tensor as F_t
except ImportError:
    from torchvision.transforms import functional as F_t
    sys.modules["torchvision.transforms.functional_tensor"] = F_t

# --- PyTorchVideo Imports ---
from pytorchvideo.transforms import (
    ApplyTransformToKey,
    UniformTemporalSubsample,
    ShortSideScale,
)
from pytorchvideo.data import make_clip_sampler, labeled_video_dataset
from pytorchvideo.models.hub import slowfast_r50
from torchvision.transforms._transforms_video import NormalizeVideo

# -------------------------------
# Settings
# -------------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

TRAIN_DIR = "/content/drive/MyDrive/fight_non_fight_model/dataset/train2"
VAL_DIR   = "/content/drive/MyDrive/fight_non_fight_model/dataset/val2"

NUM_CLASSES = 2
BATCH_SIZE = 4
EPOCHS = 12

LR = 1e-4   # ✅ Lower LR improves validation
PATIENCE = 8  # ✅ Early stopping patience

# -------------------------------
# SlowFast Pathway Helper
# -------------------------------
class PackPathway(torch.nn.Module):
    """Convert video frames into SlowFast pathways: [Slow, Fast]"""

    def forward(self, frames: torch.Tensor):
        fast_pathway = frames

        slow_pathway = torch.index_select(
            frames,
            1,
            torch.linspace(
                0, frames.shape[1] - 1, frames.shape[1] // 4
            ).long(),
        )

        return [slow_pathway, fast_pathway]

# -------------------------------
# Video Transforms
# -------------------------------
transform = ApplyTransformToKey(
    key="video",
    transform=Compose([
        UniformTemporalSubsample(32),
        Lambda(lambda x: x / 255.0),
        NormalizeVideo(mean=(0.45, 0.45, 0.45),
                       std=(0.225, 0.225, 0.225)),
        ShortSideScale(size=256),
        CenterCrop(224),
        PackPathway()
    ])
)

# -------------------------------
# Dataset & DataLoader
# -------------------------------
train_dataset = labeled_video_dataset(
    data_path=TRAIN_DIR,
    clip_sampler=make_clip_sampler("random", 2.0),
    decode_audio=False,
    transform=transform,
)

val_dataset = labeled_video_dataset(
    data_path=VAL_DIR,
    clip_sampler=make_clip_sampler("uniform", 2.0),
    decode_audio=False,
    transform=transform,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    pin_memory=True  # ✅ Important
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE
)

# -------------------------------
# Model Setup
# -------------------------------
print("Loading Pretrained SlowFast Model...")

model = slowfast_r50(pretrained=True)

# ✅ Dropout Added
model.blocks[-1].proj = torch.nn.Sequential(
    torch.nn.Dropout(0.5),
    torch.nn.Linear(2304, NUM_CLASSES)
)

model = model.to(DEVICE)

criterion = torch.nn.CrossEntropyLoss()

# ✅ AdamW + Weight Decay
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=1e-4
)

# -------------------------------
# Training Loop + Early Stopping
# -------------------------------
print("\n🚀 Training Started...\n")

best_val_acc = 0
early_stop_counter = 0

for epoch in range(EPOCHS):

    # ---- Training ----
    model.train()
    epoch_loss = 0
    correct_train, total_train = 0, 0

    for step, batch in enumerate(train_loader):

        videos = [v.to(DEVICE) for v in batch["video"]]
        labels = batch["label"].to(DEVICE)

        preds = model(videos)
        loss = criterion(preds, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

        _, predicted = preds.max(1)
        total_train += labels.size(0)
        correct_train += predicted.eq(labels).sum().item()

        if step % 10 == 0:
            acc = 100 * correct_train / total_train
            print(f"Epoch [{epoch+1}/{EPOCHS}] Step [{step}] "
                  f"| Loss: {loss.item():.4f} | Train Acc: {acc:.2f}%")

    avg_loss = epoch_loss / (step + 1)
    train_acc = 100 * correct_train / total_train

    print(f"\n✅ Epoch [{epoch+1}] Completed "
          f"| Avg Loss: {avg_loss:.4f} | Train Acc: {train_acc:.2f}%")

    # ---- Validation ----
    model.eval()
    val_loss = 0
    correct_val, total_val = 0, 0
    val_steps = 0

    with torch.no_grad():
        for batch in val_loader:

            val_steps += 1
            videos = [v.to(DEVICE) for v in batch["video"]]
            labels = batch["label"].to(DEVICE)

            preds = model(videos)
            loss = criterion(preds, labels)

            val_loss += loss.item()

            _, predicted = preds.max(1)
            total_val += labels.size(0)
            correct_val += predicted.eq(labels).sum().item()

    avg_val_loss = val_loss / val_steps
    val_acc = 100 * correct_val / total_val

    print(f"📌 Validation | Loss: {avg_val_loss:.4f} | Acc: {val_acc:.2f}%\n")

    # ---- Save Best Model ----
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        early_stop_counter = 0

        model_path = "/content/drive/MyDrive/fight_non_fight_model/slowfast_best.pt"
        torch.save(model.state_dict(), model_path)

        print(f"🏆 Best Model Saved! Val Acc: {best_val_acc:.2f}%\n")

    else:
        early_stop_counter += 1
        print(f"⚠ No improvement. EarlyStop Counter: {early_stop_counter}/{PATIENCE}\n")

    # ---- Early Stop ----
    if early_stop_counter >= PATIENCE:
        print("⛔ Early stopping triggered.")
        break

# -------------------------------
# Save Last Model
# -------------------------------
last_model_path = "/content/drive/MyDrive/fight_non_fight_model/slowfast_last.pt"
torch.save(model.state_dict(), last_model_path)

print(f"\n✅ Training Finished.")
print(f"Best Model Saved at: slowfast_best.pt")
print(f"Last Model Saved at: slowfast_last.pt")

/usr/local/lib/python3.12/dist-packages/torchvision/transforms/_functional_video.py:6: UserWarning: The 'torchvision.transforms._functional_video' module is deprecated since 0.12 and will be removed in the future. Please use the 'torchvision.transforms.functional' module instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/transforms/_transforms_video.py:22: UserWarning: The 'torchvision.transforms._transforms_video' module is deprecated since 0.12 and will be removed in the future. Please use the 'torchvision.transforms' module instead.
  warnings.warn(


Loading Pretrained SlowFast Model...

🚀 Training Started...

Epoch [1/12] Step [0] | Loss: 0.6604 | Train Acc: 50.00%
Epoch [1/12] Step [10] | Loss: 0.3564 | Train Acc: 63.64%
Epoch [1/12] Step [20] | Loss: 0.2107 | Train Acc: 73.81%
Epoch [1/12] Step [30] | Loss: 0.2694 | Train Acc: 77.42%
Epoch [1/12] Step [40] | Loss: 0.2103 | Train Acc: 78.05%

✅ Epoch [1] Completed | Avg Loss: 0.4332 | Train Acc: 79.50%
📌 Validation | Loss: 0.1385 | Acc: 100.00%

🏆 Best Model Saved! Val Acc: 100.00%

Epoch [2/12] Step [0] | Loss: 0.0580 | Train Acc: 100.00%
Epoch [2/12] Step [10] | Loss: 0.0644 | Train Acc: 95.45%
Epoch [2/12] Step [20] | Loss: 0.0679 | Train Acc: 96.43%
Epoch [2/12] Step [30] | Loss: 0.2947 | Train Acc: 95.97%
Epoch [2/12] Step [40] | Loss: 0.0555 | Train Acc: 96.95%

✅ Epoch [2] Completed | Avg Loss: 0.1589 | Train Acc: 96.00%
📌 Validation | Loss: 0.2198 | Acc: 100.00%

⚠ No improvement. EarlyStop Counter: 1/8

Epoch [3/12] Step [0] | Loss: 0.0241 | Train Acc: 100.00%
Epoch [3/1

In [ ]:
import torch
import cv2
import numpy as np
from torchvision.transforms import Compose, Lambda, CenterCrop
from pytorchvideo.transforms import (
    UniformTemporalSubsample,
    ShortSideScale,
)
from torchvision.transforms._transforms_video import NormalizeVideo
from pytorchvideo.models.hub import slowfast_r50

# --- Setup ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_PATH = "/content/drive/MyDrive/fight_non_fight_model/slowfast_best.pt"
INPUT_VIDEO = "/content/drive/MyDrive/fight_non_fight_model/dataset/video_1.mp4"
OUTPUT_VIDEO = "/content/drive/MyDrive/fight_non_fight_model/dataset/predicted_video1.mp4"

# --- Load Model ---
model = slowfast_r50(pretrained=False)
model.blocks[-1].proj = torch.nn.Sequential(
    torch.nn.Dropout(0.5),
    torch.nn.Linear(2304, 2)
)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.to(DEVICE)
model.eval()

# --- Transform Setup ---
transform = Compose([
    UniformTemporalSubsample(32),
    Lambda(lambda x: x / 255.0),
    NormalizeVideo(mean=(0.45, 0.45, 0.45), std=(0.225, 0.225, 0.225)),
    ShortSideScale(size=256),
    CenterCrop(224),
])

def pack_pathway_output(frames):
    fast_pathway = frames
    slow_pathway = torch.index_select(
        frames, 1,
        torch.linspace(0, frames.shape[1] - 1,
        frames.shape[1] // 4).long()
    )
    return [slow_pathway.to(DEVICE), fast_pathway.to(DEVICE)]

# --- Video Setup ---
cap = cv2.VideoCapture(INPUT_VIDEO)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (width, height))

frames_buffer = []

print("Processing Video with Sliding Window...")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    frames_buffer.append(rgb)

    # Maintain sliding window of 32 frames
    if len(frames_buffer) > 32:
        frames_buffer.pop(0)

    label_text = ""
    color = (0, 255, 0)  # default green

    if len(frames_buffer) == 32:
        inputs = np.array(frames_buffer).transpose(3, 0, 1, 2)
        inputs = torch.from_numpy(inputs).float()
        inputs = transform(inputs)
        inputs = pack_pathway_output(inputs)
        inputs = [i.unsqueeze(0) for i in inputs]

        with torch.no_grad():
            prediction = model(inputs)
            probs = torch.softmax(prediction, dim=1)
            conf, label_idx = torch.max(probs, dim=1)
            label_idx = label_idx.item()
            conf = conf.item()

        label_text = f"FIGHT ({conf:.2f})" if label_idx == 0 else f"NORMAL ({conf:.2f})"
        color = (0, 0, 255) if label_idx == 0 else (0, 255, 0)

        # 🔴 Draw Circle ONLY if Fight
        if label_idx == 0:
            center_coordinates = (width // 2, height // 2)
            radius = min(width, height) // 3
            cv2.circle(frame, center_coordinates, radius, (0, 0, 255), 6)

    # 🔵 Center Text Drawing
    if label_text != "":
        text_size = cv2.getTextSize(label_text,
                                    cv2.FONT_HERSHEY_SIMPLEX,
                                    1.5, 3)[0]
        text_x = (width - text_size[0]) // 2
        text_y = (height + text_size[1]) // 2

        cv2.putText(frame, label_text,
                    (text_x, text_y),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1.5, color, 3)

    out.write(frame)

cap.release()
out.release()

print(f"Done! Video saved at {OUTPUT_VIDEO}")

/usr/local/lib/python3.12/dist-packages/torchvision/transforms/_functional_video.py:6: UserWarning: The 'torchvision.transforms._functional_video' module is deprecated since 0.12 and will be removed in the future. Please use the 'torchvision.transforms.functional' module instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/transforms/_transforms_video.py:22: UserWarning: The 'torchvision.transforms._transforms_video' module is deprecated since 0.12 and will be removed in the future. Please use the 'torchvision.transforms' module instead.
  warnings.warn(


Processing Video with Sliding Window...
